# 🧪 Practice Lab: Data Wrangling for GenAI

**Module 2 · Lesson 2.1 ~90-120 min 8 exercises**  
*Netsetos GenAI Engineering Course v2.0*


**You will:**
- Convert a CSV to Parquet and measure the speed / size win

- Rewrite a Pandas eager query as a Polars lazy plan

- Build a 5-defense text cleaner that survives production text

- Truncate text token-aware (not character-aware)

- Chunk a long document with RecursiveCharacterTextSplitter + overlap

- Validate a fine-tuning JSONL with pydantic v2

- Build a 4-stage RAG ingestion pipeline on a real PDF (the centerpiece)

- Do a stratified + time-aware + group split with leakage assertions


**Prerequisites:**- Python 3.10+ (Google Colab recommended)
- Libraries: `polars`, `pyarrow`, `pandas`, `ftfy`, `tiktoken`, `pydantic`, `langchain-text-splitters`, `unstructured[pdf]`, `scikit-learn` (install via `pip` in cell 1)

---

## How to use this notebook

Each exercise shows:
- 📖 **Concept** - what you're learning
- 📝 **Task** - what to build
- A code cell with a working starter - **run it as-is first**, then modify
- 📤 **Expected output** - what you should see
- ✅ **Success criteria** - checklist to verify your work

Try every exercise. The hard ones are interview-level.

> Generated from `practice-lab-2_1.html` - re-run the generator if the lab updates.


In [ ]:
# Practice lab setup
# Colab: install the non-default packages the exercise cells import.
!pip install -q polars ftfy tiktoken langchain-text-splitters "unstructured[pdf]"

# Each exercise has its own imports - cells are independent.
# Run cells in order, modify them to experiment.

### 🎯 1. CSV to Parquet - measure the win

`Easy` · `~10 min`

**📖 Concept**

The format choice IS the architecture choice. Parquet is columnar, compressed, schema-encoded. The same 1M-row dataset is ~5-40x smaller on disk and ~10-50x faster to load. This is your highest-leverage refactor in any data pipeline.

**📝 Task**

1. Generate a 1M-row synthetic dataset (5 columns: int, float, string, bool, timestamp).  

    2. Write it as CSV and as Parquet. Measure file size of each.  

    3. Time the read of each format. Report load time + RAM.  

    4. Run the same filter+groupby query on both. Report query time.  

    5. Print a comparison table: format / size / load time / query time.

In [ ]:
# Exercise: 1. CSV to Parquet - measure the win
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

import polars as pl
import numpy as np, os, time
from datetime import datetime, timedelta

N = 1_000_000
df = pl.DataFrame({
    "user_id":  np.random.randint(1, 100_000, N),
    "amount":   np.random.gamma(2, 50, N).round(2),
    "event":    np.random.choice(["purchase","view","signup","logout"], N),
    "active":   np.random.choice([True, False], N),
    "ts":       [datetime(2026,1,1)+timedelta(seconds=int(s))
                 for s in np.random.randint(0, 86400*30, N)],
})

# TODO 1: write df.write_csv("events.csv") and df.write_parquet("events.parquet")
# TODO 2: record os.path.getsize for both
# TODO 3: time pl.read_csv vs pl.read_parquet (run each 3x, take min)
# TODO 4: same .filter(pl.col("event")=="purchase").group_by("user_id").agg(pl.col("amount").sum()).collect() on each
# TODO 5: print comparison table

**📤 Expected output**

```
Format    Size (MB)  Load (s)  Query (s)
CSV       58.4       1.820     0.412
Parquet    4.7       0.061     0.038

Parquet is 12.4x smaller, 29.8x faster to load, 10.8x faster to query.
```

**✅ Success criteria**

- [ ] 1M rows written in both formats
- [ ] File sizes printed; Parquet is ~10-15x smaller
- [ ] Load times printed; Parquet is ~10-30x faster
- [ ] Query times printed; Parquet wins via lazy plan + columnar reads

### 🎯 2. Pandas eager to Polars lazy

`Easy` · `~10 min`

**📖 Concept**

Pandas runs each operation immediately and materializes a new DataFrame. Polars lazy mode builds a query plan and optimizes it before executing: pushes filters into the Parquet read, prunes unused columns, re-orders operations, parallelizes across cores.

**📝 Task**

1. Re-use the Parquet from Exercise 1.  

    2. Write the same query in Pandas (eager) and Polars lazy.  

    3. Print the optimized plan with `q.explain()` - identify which optimizations Polars applied.  

    4. Time both. Report the speedup.  

    5. Identify which two columns Polars pruned at read time.

In [ ]:
# Exercise: 2. Pandas eager to Polars lazy
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

import polars as pl
import pandas as pd
import time

# Pandas eager
def pandas_query():
    df = pd.read_parquet("events.parquet")
    df = df[df["event"] == "purchase"]
    df = df[df["amount"] > 100]
    return df.groupby("user_id")["amount"].sum().reset_index()

# Polars lazy
def polars_lazy():
    return (pl.scan_parquet("events.parquet")
              .filter(pl.col("event") == "purchase")
              .filter(pl.col("amount") > 100)
              .group_by("user_id")
              .agg(pl.col("amount").sum())
              .collect())

# TODO 1: time both (best of 3)
# TODO 2: print pl.scan_parquet(...)....explain() - look for "FILTER" pushed down
# TODO 3: count columns Polars actually reads vs Pandas (Pandas reads all 5)

**📤 Expected output**

```
Pandas eager: 0.412s
Polars lazy:  0.038s   (10.8x faster)

Polars plan (optimized):
  AGGREGATE [SUM(amount)] BY [user_id]
   FROM SELECTION: [(event=="purchase") AND (amount>100)]
    FROM PARQUET SCAN events.parquet
     PROJECT 3/5 COLUMNS  <-- pruned bool + ts at read time
     FILTER predicate pushdown applied
```

**✅ Success criteria**

- [ ] Pandas and Polars versions produce same result
- [ ] Polars runs 5-30x faster on this workload
- [ ] q.explain() shows predicate pushdown + column pruning
- [ ] You can name the two columns Polars skipped reading (active, ts)

### 🎯 3. The 5-defense text cleaner

`Easy` · `~10 min`

**📖 Concept**

Production text is hostile. Real strings from APIs, scrapers, and user input contain mojibake, BOMs, zero-width characters, PII, and exceed token limits. The 5 defenses, layered in order, are the difference between fine-tuning that converges and fine-tuning that learns garbage.

**📝 Task**

1. Implement `clean(s, max_tokens=2000) -> str` with all 5 defenses in the correct order.  

    2. Build a 4-row "hostile inputs" test set: mojibake, BOM, zero-width chars, PII.  

    3. Run each test row through the cleaner and assert the expected output.  

    4. Print before / after side-by-side for each row.  

    5. Add a 5th test row that exceeds max_tokens to verify token truncation.

In [ ]:
# Exercise: 3. The 5-defense text cleaner
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

import ftfy, unicodedata, re, tiktoken
enc = tiktoken.encoding_for_model("gpt-4")

def clean(s: str, max_tokens: int = 2000) -> str:
    # TODO 1: fix mojibake via ftfy.fix_text
    # TODO 2: NFKC normalize
    # TODO 3: strip zero-width chars (U+200B, U+200C, U+200D, U+FEFF)
    # TODO 4: redact emails (and add: phone numbers if you want a bonus)
    # TODO 5: token-truncate via tiktoken
    pass

tests = [
    "RobotsKommen heute",           # mojibake
    "﻿hello world",                       # BOM
    "this​ has‌ zw‍ chars",     # zero-width
    "Contact me at user@example.com please",   # PII
    "x" * 30_000,                              # too long
]
# TODO: print before/after for each, plus assertion that the result has 0 zero-width
#       chars and at most max_tokens tokens.

**📤 Expected output**

```
[mojibake]   in: 'RobotsKommen heute'
             out: 'RobotsKommen heute'
[BOM]        in: '﻿hello world'
             out: 'hello world'
[zero-width] in: 'this​ has‌ zw‍ chars'
             out: 'this has zw chars'
[PII]        in: 'Contact me at user@example.com please'
             out: 'Contact me at [EMAIL] please'
[long]       in:  30000 chars
             out: 2000 tokens (~7000 chars after truncation)

All assertions passed.
```

**✅ Success criteria**

- [ ] All 5 defenses implemented in the correct order
- [ ] Mojibake repaired (German umlauts visible)
- [ ] BOM removed
- [ ] Zero-width chars stripped (regex hit-count = 0 in output)
- [ ] Email redacted to [EMAIL]
- [ ] Long input truncated to exactly max_tokens tokens

### 🎯 4. Token-aware truncation

`Medium` · `~12 min`

**📖 Concept**

Characters and tokens are not the same. A 4000-character string is ~800 tokens in English, ~2000 in Hindi, ~3000 in Japanese. Truncating by character count silently corrupts inputs in non-Latin scripts. Always count tokens.

**📝 Task**

1. Build a 3-language test set: English, Hindi, Japanese (~100 words each).  

    2. For each, compute char count, token count (gpt-4 tokenizer), and chars-per-token.  

    3. Truncate each string by character to `len // 2` AND by token to `tokens // 2`. Compare the two outputs.  

    4. Print a table showing how character truncation under-cuts non-Latin scripts.  

    5. Write a helper `truncate_by_tokens(s, n)` that always cuts at exactly n tokens.

In [ ]:
# Exercise: 4. Token-aware truncation
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

import tiktoken
enc = tiktoken.encoding_for_model("gpt-4")

samples = {
    "english": "The quick brown fox jumps over the lazy dog. " * 12,
    "hindi":   "tej bhura lomdee aalsee kutte ke upar koodatee hai. " * 12,  # (transliterated; use Devanagari in your run)
    "japanese":"sokuyaku no chairo no kitsune ga namake mono no inu wo tobikoeru. " * 8,
}

def truncate_by_tokens(s: str, n: int) -> str:
    # TODO: encode, slice to n, decode
    pass

# TODO: for each language print char_count, token_count, chars_per_token
# TODO: compare char-truncation (s[:len(s)//2]) vs token-truncation
# TODO: show that char-truncation cuts MORE meaning out of Hindi/Japanese

**📤 Expected output**

```
english : chars= 540 tokens= 132  chars/tok=4.09
hindi   : chars= 624 tokens= 295  chars/tok=2.12
japanese: chars= 624 tokens= 410  chars/tok=1.52

Char-truncation effect:
  english :  s[:270] keeps ~66 tokens (50% of tokens)
  hindi   :  s[:312] keeps ~148 tokens (50%) - okay-ish
  japanese:  s[:312] keeps ~205 tokens (50%) - okay-ish

Token-truncation effect (target = half the tokens):
  english :  truncate_by_tokens(s, 66)  -> 268 chars (correct)
  hindi   :  truncate_by_tokens(s, 147) -> 311 chars (correct)
  japanese:  truncate_by_tokens(s, 205) -> 311 chars (correct)

Use token-truncation when feeding LLM APIs.
```

**✅ Success criteria**

- [ ] 3 languages tested with tiktoken
- [ ] chars-per-token varies meaningfully across languages (~4 for English, ~2 for Hindi, ~1.5 for Japanese)
- [ ] truncate_by_tokens helper works for all three
- [ ] You can explain why character truncation is broken for non-Latin scripts

### 🎯 5. Chunking with RecursiveCharacterTextSplitter

`Medium` · `~15 min`

**📖 Concept**

RAG chunking is the single highest-leverage knob in retrieval quality. RecursiveCharacterTextSplitter tries paragraph-break, then sentence-break, then word-break, then character-break in order so chunks rarely cut mid-sentence. The 512-token chunk with 64-token overlap is the modern default, but the right setting is workload-specific.

**📝 Task**

1. Build a ~2500-token text blob (any long article or use the embedded example).  

    2. Chunk it three ways: (chunk=256, overlap=0), (chunk=512, overlap=64), (chunk=1024, overlap=128).  

    3. For each setting: print chunk count, average chunk tokens, max chunk tokens, total overlap tokens.  

    4. Find one chunk pair and print the overlap region (the shared tail of one and head of next).  

    5. Explain: which setting has the LOWEST risk of cutting mid-sentence?

In [ ]:
# Exercise: 5. Chunking with RecursiveCharacterTextSplitter
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4")

text = """Polars is a DataFrame library written in Rust...""" * 50  # build ~2500 tokens

def chunk(text, size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        length_function=lambda s: len(enc.encode(s)),
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)

# TODO 1: chunk at (256,0), (512,64), (1024,128)
# TODO 2: print count, mean tokens, max tokens
# TODO 3: extract the overlap region between chunks[0] and chunks[1]
# TODO 4: report which config has the lowest fraction of chunks ending mid-sentence

**📤 Expected output**

```
size= 256 overlap=  0 n=11 mean=242 max=256 mid-sentence-ends=3/11
size= 512 overlap= 64 n= 6 mean=487 max=512 mid-sentence-ends=1/6
size=1024 overlap=128 n= 3 mean=910 max=1024 mid-sentence-ends=0/3

Overlap region between chunks[0] and chunks[1] at (512,64):
  ...last 64 tokens of chunk 0 == first 64 tokens of chunk 1

The bigger chunks have lower mid-sentence-cut risk but worse retrieval precision.
512+64 is the modern default - balanced precision vs cut-safety.
```

**✅ Success criteria**

- [ ] 3 chunking configurations compared
- [ ] Token counts (mean / max) reported per config
- [ ] Overlap region between two chunks extracted and verified
- [ ] You can articulate the precision / recall / cut-safety tradeoff

### 🎯 6. pydantic schema for fine-tuning JSONL

`Medium` · `~12 min`

**📖 Concept**

Fine-tuning APIs validate JSON shape. They do NOT validate empty content, token limits, or duplicates. Each of those silently wastes ~$50-$500 per run. A 30-second pydantic pass catches them all before `files.create`.

**📝 Task**

1. Define a `FineTuneRow` pydantic v2 model with these validations:  

       a) Must have at least one user and one assistant message  

       b) Last message must be assistant with non-empty content  

       c) Total tokens (sum of all messages) ≤ 16000 for gpt-4 fine-tuning  

    2. Validate a synthetic 100-row JSONL where you intentionally seed 8 bad rows: 2 empty, 3 too-long, 2 missing roles, 1 duplicate.  

    3. Print row-level errors and a summary: valid / bad by category.  

    4. Write the cleaned (valid-only) JSONL to a new file.  

    5. Bonus: detect duplicates via content hash.

In [ ]:
# Exercise: 6. pydantic schema for fine-tuning JSONL
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

from pydantic import BaseModel, field_validator, ValidationError
import json, hashlib
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4")

class Msg(BaseModel):
    role: str
    content: str

class FineTuneRow(BaseModel):
    messages: list[Msg]

    @field_validator("messages")
    @classmethod
    def validate(cls, v):
        # TODO a: at least 1 user + 1 assistant
        # TODO b: last is assistant with non-empty content
        # TODO c: total tokens <= 16000
        return v

# TODO: build a 100-row synthetic JSONL with 8 seeded bad rows
# TODO: iterate, validate, collect errors by category
# TODO: write valid-only rows to ft_clean.jsonl
# TODO bonus: dedupe by hashlib.md5(json.dumps(messages, sort_keys=True))

**📤 Expected output**

```
Row   3: ValidationError - last msg must be non-empty assistant
Row  17: ValidationError - too long: 17234 tokens
Row  21: ValidationError - missing roles (assistant absent)
Row  44: ValidationError - too long: 19891 tokens
Row  58: ValidationError - last msg must be non-empty assistant
Row  72: ValidationError - missing roles (user absent)
Row  88: ValidationError - too long: 16412 tokens
Row  91: duplicate of row 14

Summary: 92 valid / 8 bad
  empty:      2
  too long:   3
  bad roles:  2
  duplicate:  1

Wrote 92 rows to ft_clean.jsonl. Saved ~$45 of GPU time.
```

**✅ Success criteria**

- [ ] pydantic v2 model with 3 validators
- [ ] All 8 seeded bad rows caught with the right error category
- [ ] Valid rows written to a clean JSONL
- [ ] Duplicate detection via content hash (bonus)

### 🎯 7. The full 4-stage RAG ingestion pipeline (centerpiece)

`Hard` · `~25 min`

**📖 Concept**

This is the lesson's deliverable. Take a real 2-column PDF (an arXiv paper works), parse with unstructured.io, filter out headers/footers, chunk to 512+64 tokens, and write embed-ready JSONL with metadata. Every RAG system in Module 8 starts here.

**📝 Task**

1. Download a real arXiv PDF (any recent one - prefer 2-column LaTeX).  

    2. **Stage 1 - parse:** run `partition_pdf(strategy="hi_res")` and count elements by type.  

    3. **Stage 2 - filter:** keep only NarrativeText, Title, ListItem, Table. Report how many tokens were dropped.  

    4. **Stage 3 - chunk:** use RecursiveCharacterTextSplitter at 512+64 with token-based length.  

    5. **Stage 4 - serialize:** write embed-ready JSONL with fields: `text`, `source`, `page`, `element_type`, `chunk_idx`.  

    6. As a sanity check, retrieve the first chunk back, print it, confirm it reads as a coherent paragraph (not column-mixed).  

    7. Compare PyPDF2 output on the same PDF and note how broken it is.

In [ ]:
# Exercise: 7. The full 4-stage RAG ingestion pipeline (centerpiece)
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

# pip install "unstructured[pdf]" langchain-text-splitters pypdf2

from unstructured.partition.pdf import partition_pdf
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken, json, requests
enc = tiktoken.encoding_for_model("gpt-4")

# Download a sample PDF (replace with any recent arXiv URL)
url = "https://arxiv.org/pdf/2402.19473.pdf"  # any 2-column paper
with open("paper.pdf","wb") as f:
    f.write(requests.get(url).content)

# Stage 1 - parse
elements = partition_pdf("paper.pdf", strategy="hi_res")
from collections import Counter
print("Element types:", Counter(e.category for e in elements))

# Stage 2 - filter
KEEP = {"NarrativeText", "Title", "ListItem", "Table"}
filtered = [e for e in elements if e.category in KEEP]
before_tok = sum(len(enc.encode(e.text)) for e in elements)
after_tok = sum(len(enc.encode(e.text)) for e in filtered)
print(f"Tokens before: {before_tok}, after: {after_tok}, dropped: {before_tok-after_tok}")

# Stage 3 - chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64,
    length_function=lambda s: len(enc.encode(s)),
)
chunks = []
for e in filtered:
    for i, ch in enumerate(splitter.split_text(e.text)):
        chunks.append({
            "text": ch,
            "source": "paper.pdf",
            "page": getattr(e.metadata, "page_number", None),
            "element_type": e.category,
            "chunk_idx": len(chunks),
        })

# Stage 4 - serialize
with open("paper_chunks.jsonl", "w") as f:
    for c in chunks:
        f.write(json.dumps(c) + "\n")
print(f"Wrote {len(chunks)} chunks to paper_chunks.jsonl")
print("First chunk preview:", chunks[0]["text"][:200])

# TODO: also run PyPDF2 on the same PDF and print the first 500 chars - compare quality

**📤 Expected output**

```
Element types: Counter({'NarrativeText': 142, 'Title': 18, 'Header': 24,
                         'Footer': 14, 'ListItem': 11, 'Table': 6, 'Image': 3})

Tokens before: 14,820   after: 11,562   dropped: 3,258 (~22%)
Wrote 38 chunks to paper_chunks.jsonl

First chunk preview:
"A Survey of RAG Systems. We present a comprehensive overview of retrieval-
augmented generation systems, focusing on the four-stage ingestion pipeline:
parse, filter, chunk, embed. The dominant production framework is..."
   <-- reads as coherent paragraph, not column-mixed!

PyPDF2 first 500 chars:
"A Survey of RAG Systems Department of CS We present XYZ University a
comprehensive 1 Introduction overview Foundational of retrieval models..."
   <-- columns interleaved, unreadable
```

**✅ Success criteria**

- [ ] Real PDF parsed via unstructured.io with element types counted
- [ ] Filter step drops ~15-35% of tokens (junk)
- [ ] Chunked output is paragraph-coherent, not column-mixed
- [ ] JSONL has source / page / element_type / chunk_idx metadata
- [ ] PyPDF2 comparison demonstrates the column-mixing failure

### 🎯 8. Leakage-safe splits: stratified + time-aware + group

`Hard` · `~20 min`

**📖 Concept**

Random_state=42 doesn't fix bad splits - it makes them reproducible. Match the splitter to the data's structure: stratify for imbalanced classes, time-aware for time-series, group-aware for correlated rows. Then ASSERT no leakage before training.

**📝 Task**

1. Generate a synthetic dataset of 5000 support tickets with: `ticket_id`, `user_id` (200 unique), `created_at` (Jan-Dec 2024), `category` (95% routine, 5% escalation), `text`.  

    2. Do THREE splits in sequence:  

       a) Stratified 80/10/10 on `category` - assert class proportions match across splits  

       b) Time-aware: train = Jan-Sep, val = Oct, test = Nov-Dec - assert no timestamp overlap  

       c) GroupKFold on `user_id` - assert no user_id appears in both train and val  

    3. For each split, print: row counts, class distribution, date range, group overlap.  

    4. Show what BREAKS with a naive random split (run it, compare violations).

In [ ]:
# Exercise: 8. Leakage-safe splits: stratified + time-aware + group
# Run this cell first - it's a complete working example.
# Then modify it to experiment.

import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from datetime import datetime, timedelta

np.random.seed(42)
N = 5000
df = pd.DataFrame({
    "ticket_id": range(N),
    "user_id":   np.random.randint(0, 200, N),
    "created_at": [datetime(2024,1,1)+timedelta(hours=int(h))
                   for h in np.sort(np.random.randint(0, 8760, N))],
    "category":  np.random.choice(["routine","escalation"], N, p=[0.95, 0.05]),
})

# a) Stratified split
def stratified_split(df):
    train, temp = train_test_split(df, test_size=0.20, stratify=df["category"], random_state=42)
    val, test = train_test_split(temp, test_size=0.50, stratify=temp["category"], random_state=42)
    # TODO: assert class proportions match across all 3
    return train, val, test

# b) Time-aware split
def time_split(df):
    df = df.sort_values("created_at")
    # TODO: train = before Oct, val = Oct, test = Nov+
    # TODO: assert train.created_at.max() < val.created_at.min() etc
    pass

# c) Group split
def group_split(df, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    # TODO: get first fold, assert set(train.user_id) & set(val.user_id) == set()
    pass

# Naive random for comparison
def random_split(df):
    return train_test_split(df, test_size=0.20, random_state=42)

# TODO: run all 4, print comparison table

**📤 Expected output**

```
Stratified 80/10/10:
  train: 4000 rows, escalation=4.95%, dates Jan 2024 - Dec 2024
  val:    500 rows, escalation=5.00%, dates Jan 2024 - Dec 2024
  test:   500 rows, escalation=5.20%, dates Jan 2024 - Dec 2024
  OK class proportions match (max delta = 0.25%)

Time-aware:
  train: 3742 rows, dates Jan - Sep 30
  val:    421 rows, dates Oct 1 - Oct 31
  test:   837 rows, dates Nov 1 - Dec 31
  OK no timestamp overlap

GroupKFold (5 folds, fold 0):
  train: 4000 rows, 160 unique users
  val:   1000 rows,  40 unique users
  user overlap: 0
  OK no user leakage

Naive random split (for comparison):
  user_id overlap: 199/200 (99.5%)  <-- LEAKAGE!
  Time range: train and val both span Jan - Dec  <-- LEAKAGE!
  Random_state=42 just makes this bad split reproducible.
```

**✅ Success criteria**

- [ ] 3 split strategies implemented (stratified, time-aware, group)
- [ ] All assertions pass: no class imbalance, no timestamp overlap, no group leakage
- [ ] Naive random split is shown to violate all 3 invariants
- [ ] You can articulate which strategy fits which data structure

---

## 🪞 Reflection

**Reflect:** You have now shipped the data-wrangling toolkit a 2026 GenAI engineer actually needs. The 4-stage RAG ingestion pipeline (Ex 7) reappears in Module 8. The pydantic + JSONL validation (Ex 6) reappears in Module 9 (fine-tuning). The time-aware split (Ex 8) reappears in Module 14 (LLMOps eval-set rotation). Five techniques, reused everywhere downstream.

---

## 🎓 Practice complete

You've worked through the practice exercises for **Practice Lab: Data Wrangling for GenAI**.

### Next:
- Compare your solutions to the reference code in each cell
- Modify the code to experiment with different inputs
- Move on to the next lesson

*Generated from `practice-lab-2_1.html` - re-run the generator if the lab updates.*
